In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

import pingouin as pg
import os
from tqdm import tqdm
import plotly.graph_objects as go

from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet

warnings.filterwarnings('ignore')
# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
df = pd.read_excel(r'E:\pycharm all files\眼动数据处理\飞行得分预测\data_clean.xlsx')
# Alcohol
df = df[df['组别'] == 0]
df

,组别,受试者,session,SCL_mean,AOI_Transition,SGE,GTE,HR,SDNN,RMSSD,...,Cz_α/β_low,Cz_θ/β_low,Fz_α/β_low,Fz_θ/β_low,Pz_α/β_low,Pz_θ/β_low,DE_Cz_β_low,DE_Fz_β_low,DE_Pz_β_low,label
0,0,付瑞晗,1,0.008423,5.964286,0.793663,0.297438,17.819101,-3.260506,1.857342,...,-1.532164,-1.373074,-2.761845,-2.716964,1.572004,1.408181,0.117051,0.080689,0.190822,0
1,0,付瑞晗,2,0.008640,10.500000,0.884336,0.313038,5.119280,-15.197881,-21.082419,...,2.106585,1.667920,1.392895,1.070889,5.278246,4.543939,0.530528,0.276095,0.148971,0
2,0,付瑞晗,3,0.008858,15.035714,0.975010,0.328639,9.814463,11.623381,13.436624,...,1.328775,1.317241,1.174340,1.011112,0.795306,0.452237,-0.125163,0.177508,0.124151,2
3,0,付瑞晗,4,0.009075,19.571429,1.065683,0.344240,44.991362,25.830811,31.173141,...,0.593283,0.116811,1.264330,0.899754,0.603882,0.877743,0.392523,0.244415,0.253055,2
4,0,付瑞晗,5,0.009293,24.107143,1.156356,0.359840,10.378346,33.789581,41.231056,...,0.828338,0.724440,0.786245,0.561229,1.452079,0.498206,0.028311,0.136193,0.206192,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,0,黄博文,3,0.004806,11.345238,1.063506,0.321262,-2.306180,58.790809,35.181470,...,3.018931,1.951814,0.380273,-0.327464,0.494465,0.165513,0.164719,-0.075142,-0.234687,1
108,0,黄博文,4,0.004377,11.142857,1.067830,0.290355,-76.143064,27.028106,12.410315,...,-1.329368,2.356575,4.306484,2.031069,0.664819,1.724746,0.218595,0.145073,0.209449,0
109,0,黄博文,5,0.003948,10.940476,1.072154,0.259449,-0.210315,68.831714,87.335560,...,-0.199533,-0.070551,-0.907792,-1.865179,2.817726,-1.519484,-0.084545,0.331388,0.288583,3
110,0,黄博文,6,0.003519,10.738095,1.076479,0.228543,-8.138341,97.847874,77.089929,...,11.347848,17.104127,1.869395,1.813472,2.068649,1.984400,-0.103697,0.117443,0.222504,3


In [3]:
X_cols = [
    'SCL_mean', 'AOI_Transition', 'SGE', 'GTE',
    'HR', 'SDNN', 'RMSSD',
    'Cz_β_high', 'Fz_β_high', 'Pz_β_high',
    'Cz_α/β_high', 'Cz_θ/β_high',
    'Fz_α/β_high', 'Fz_θ/β_high',
    'Pz_α/β_high', 'Pz_θ/β_high',
    'DE_Cz_β_high', 'DE_Fz_β_high', 'DE_Pz_β_high',
    'Cz_β_low', 'Fz_β_low', 'Pz_β_low',
    'Cz_α/β_low', 'Cz_θ/β_low',
    'Fz_α/β_low', 'Fz_θ/β_low',
    'Pz_α/β_low', 'Pz_θ/β_low',
    'DE_Cz_β_low', 'DE_Fz_β_low', 'DE_Pz_β_low'
]

# 可选：把组别和 session 也加进去
X_cols += ['session']

X = df[X_cols]
y = df['label']
groups = df['受试者']


In [4]:
X

,SCL_mean,AOI_Transition,SGE,GTE,HR,SDNN,RMSSD,Cz_β_high,Fz_β_high,Pz_β_high,...,Cz_α/β_low,Cz_θ/β_low,Fz_α/β_low,Fz_θ/β_low,Pz_α/β_low,Pz_θ/β_low,DE_Cz_β_low,DE_Fz_β_low,DE_Pz_β_low,session
0,0.008423,5.964286,0.793663,0.297438,17.819101,-3.260506,1.857342,-0.729108,-0.440792,1.180175,...,-1.532164,-1.373074,-2.761845,-2.716964,1.572004,1.408181,0.117051,0.080689,0.190822,1
1,0.008640,10.500000,0.884336,0.313038,5.119280,-15.197881,-21.082419,0.599527,1.112453,0.374152,...,2.106585,1.667920,1.392895,1.070889,5.278246,4.543939,0.530528,0.276095,0.148971,2
2,0.008858,15.035714,0.975010,0.328639,9.814463,11.623381,13.436624,0.133105,1.244384,0.784893,...,1.328775,1.317241,1.174340,1.011112,0.795306,0.452237,-0.125163,0.177508,0.124151,3
3,0.009075,19.571429,1.065683,0.344240,44.991362,25.830811,31.173141,1.249114,1.324410,0.054951,...,0.593283,0.116811,1.264330,0.899754,0.603882,0.877743,0.392523,0.244415,0.253055,4
4,0.009293,24.107143,1.156356,0.359840,10.378346,33.789581,41.231056,1.462762,1.520320,0.673478,...,0.828338,0.724440,0.786245,0.561229,1.452079,0.498206,0.028311,0.136193,0.206192,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,0.004806,11.345238,1.063506,0.321262,-2.306180,58.790809,35.181470,0.032903,0.955599,0.802129,...,3.018931,1.951814,0.380273,-0.327464,0.494465,0.165513,0.164719,-0.075142,-0.234687,3
108,0.004377,11.142857,1.067830,0.290355,-76.143064,27.028106,12.410315,0.872509,1.036521,0.255333,...,-1.329368,2.356575,4.306484,2.031069,0.664819,1.724746,0.218595,0.145073,0.209449,4
109,0.003948,10.940476,1.072154,0.259449,-0.210315,68.831714,87.335560,0.520915,0.244861,0.404265,...,-0.199533,-0.070551,-0.907792,-1.865179,2.817726,-1.519484,-0.084545,0.331388,0.288583,5
110,0.003519,10.738095,1.076479,0.228543,-8.138341,97.847874,77.089929,1.002349,0.963907,1.179298,...,11.347848,17.104127,1.869395,1.813472,2.068649,1.984400,-0.103697,0.117443,0.222504,6


In [5]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=8)  # 32人 → 8折，每折4人


# 采用ElasticNet模型

In [6]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNet(max_iter=10000))
])

param_grid = {
    'model__alpha': np.logspace(-3, 1, 20),
    'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

gkf = GroupKFold(n_splits=8)

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=gkf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid.fit(X, y, groups=groups)

baseline_rmse = mean_squared_error(
    y,
    np.full_like(y, y.mean()),
    squared=False
)

print("最优参数:", grid.best_params_)
print("CV RMSE:", -grid.best_score_)
print("Baseline RMSE:", baseline_rmse)

最优参数: {'model__alpha': 0.20691380811147903, 'model__l1_ratio': 0.9}
CV RMSE: 1.1830920368282967
Baseline RMSE: 1.6447969913813507


# 被试级预测性能评估（不是 session 级）

In [7]:
from sklearn.model_selection import cross_val_predict

y_pred = cross_val_predict(
    grid.best_estimator_,
    X, y,
    cv=gkf,
    groups=groups
)

rmse = mean_squared_error(y, y_pred, squared=False)
r2 = r2_score(y, y_pred)

print("被试级 RMSE:", rmse)
print("R²:", r2)


被试级 RMSE: 1.1863921187728084
R²: 0.19945547952866416


In [8]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np


def group_cv_rmse_r2(estimator, X, y, groups, n_splits=8):
    gkf = GroupKFold(n_splits=n_splits)

    y_true_all = []
    y_pred_all = []

    for train_idx, test_idx in gkf.split(X, y, groups):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        estimator.fit(X_train, y_train)
        y_pred = estimator.predict(X_test)

        y_true_all.append(y_test.values)
        y_pred_all.append(y_pred)

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)

    rmse = mean_squared_error(y_true_all, y_pred_all, squared=False)
    r2 = r2_score(y_true_all, y_pred_all)

    return rmse, r2


In [9]:
from sklearn.linear_model import Ridge

ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge())
])

ridge_param = {
    'model__alpha': np.logspace(-3, 3, 20)
}

ridge_grid = GridSearchCV(
    ridge_pipe,
    ridge_param,
    cv=gkf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

ridge_grid.fit(X, y, groups=groups)


GridSearchCV(cv=GroupKFold(n_splits=8),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', Ridge())]),
             n_jobs=-1,
             param_grid={'model__alpha': array([1.00000000e-03, 2.06913808e-03, 4.28133240e-03, 8.85866790e-03,
       1.83298071e-02, 3.79269019e-02, 7.84759970e-02, 1.62377674e-01,
       3.35981829e-01, 6.95192796e-01, 1.43844989e+00, 2.97635144e+00,
       6.15848211e+00, 1.27427499e+01, 2.63665090e+01, 5.45559478e+01,
       1.12883789e+02, 2.33572147e+02, 4.83293024e+02, 1.00000000e+03])},
             scoring='neg_root_mean_squared_error')

In [10]:
from sklearn.linear_model import Lasso

lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(max_iter=10000))
])

lasso_param = {
    'model__alpha': np.logspace(-4, 1, 20)
}

lasso_grid = GridSearchCV(
    lasso_pipe,
    lasso_param,
    cv=gkf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

lasso_grid.fit(X, y, groups=groups)


GridSearchCV(cv=GroupKFold(n_splits=8),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', Lasso(max_iter=10000))]),
             n_jobs=-1,
             param_grid={'model__alpha': array([1.00000000e-04, 1.83298071e-04, 3.35981829e-04, 6.15848211e-04,
       1.12883789e-03, 2.06913808e-03, 3.79269019e-03, 6.95192796e-03,
       1.27427499e-02, 2.33572147e-02, 4.28133240e-02, 7.84759970e-02,
       1.43844989e-01, 2.63665090e-01, 4.83293024e-01, 8.85866790e-01,
       1.62377674e+00, 2.97635144e+00, 5.45559478e+00, 1.00000000e+01])},
             scoring='neg_root_mean_squared_error')

In [11]:
from sklearn.svm import SVR

svr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVR(kernel='rbf'))
])

svr_param = {
    'model__C': np.logspace(-2, 2, 5),
    'model__gamma': np.logspace(-3, 0, 5)
}

svr_grid = GridSearchCV(
    svr_pipe,
    svr_param,
    cv=gkf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

svr_grid.fit(X, y, groups=groups)


GridSearchCV(cv=GroupKFold(n_splits=8),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', SVR())]),
             n_jobs=-1,
             param_grid={'model__C': array([1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02]),
                         'model__gamma': array([0.001     , 0.00562341, 0.03162278, 0.17782794, 1.        ])},
             scoring='neg_root_mean_squared_error')

In [12]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_param = {
    'model__max_depth': [3, 5, 10, None],
    'model__min_samples_leaf': [1, 3, 5]
}

rf_pipe = Pipeline([
    ('model', rf)
])

rf_grid = GridSearchCV(
    rf_pipe,
    rf_param,
    cv=gkf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

rf_grid.fit(X, y, groups=groups)


GridSearchCV(cv=GroupKFold(n_splits=8),
             estimator=Pipeline(steps=[('model',
                                        RandomForestRegressor(n_estimators=300,
                                                              n_jobs=-1,
                                                              random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [3, 5, 10, None],
                         'model__min_samples_leaf': [1, 3, 5]},
             scoring='neg_root_mean_squared_error')

In [13]:
enet_rmse, enet_r2 = group_cv_rmse_r2(
    grid.best_estimator_, X, y, groups
)
ridge_rmse, ridge_r2 = group_cv_rmse_r2(
    ridge_grid.best_estimator_, X, y, groups
)
lasso_rmse, lasso_r2 = group_cv_rmse_r2(
    lasso_grid.best_estimator_, X, y, groups
)
svr_rmse, svr_r2 = group_cv_rmse_r2(
    svr_grid.best_estimator_, X, y, groups
)
rf_rmse, rf_r2 = group_cv_rmse_r2(
    rf_grid.best_estimator_, X, y, groups
)

In [14]:
y_pred_base = np.full_like(y, y.mean())
baseline_rmse = mean_squared_error(y, y_pred_base, squared=False)
baseline_r2 = r2_score(y, y_pred_base)

print(f"{'Baseline':10s}  RMSE = {baseline_rmse},r2 = {baseline_r2}")  # r2 ≈ 0

Baseline    RMSE = 1.6447969913813507,r2 = -0.5386987077760146


In [15]:
results = {
    'ElasticNet': (enet_rmse, enet_r2),
    'Ridge': (ridge_rmse, ridge_r2),
    'Lasso': (lasso_rmse, lasso_r2),
    'SVR (RBF)': (svr_rmse, svr_r2),
    'Random Forest': (rf_rmse, rf_r2),
    'Baseline': (baseline_rmse, baseline_r2)
}

for model, (rmse, r2) in results.items():
    print(f"{model:15s}  RMSE = {rmse:.3f} | R² = {r2:.3f}")


ElasticNet       RMSE = 1.186 | R² = 0.199
Ridge            RMSE = 1.338 | R² = -0.019
Lasso            RMSE = 1.189 | R² = 0.195
SVR (RBF)        RMSE = 1.283 | R² = 0.064
Random Forest    RMSE = 1.218 | R² = 0.157
Baseline         RMSE = 1.645 | R² = -0.539


# 可选增强（如果你想再榨一点性能）

## 方案 A：PCA + Ridge（最稳、最推荐）

In [16]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

pipe_pca_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('model', Ridge())
])

param_grid = {
    'pca__n_components': [0.8, 0.9, 0.95],
    'model__alpha': np.logspace(-3, 3, 20)
}

gkf = GroupKFold(n_splits=8)

grid_pca_ridge = GridSearchCV(
    pipe_pca_ridge,
    param_grid,
    cv=gkf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_pca_ridge.fit(X, y, groups=groups)

print("最优参数:", grid_pca_ridge.best_params_)
print("CV RMSE:", -grid_pca_ridge.best_score_)

y_pred = cross_val_predict(
    grid_pca_ridge.best_estimator_,
    X, y,
    cv=gkf,
    groups=groups
)

print("被试级 RMSE:", mean_squared_error(y, y_pred, squared=False))
print("R²:", r2_score(y, y_pred))


最优参数: {'model__alpha': 483.2930238571752, 'pca__n_components': 0.95}
CV RMSE: 1.324393489057508
被试级 RMSE: 1.33351877907268
R²: -0.011410027632998254


## 方案 B：按模态分组 PCA（更高级、更“论文感”）

In [17]:
eye_features = ['AOI_Transition', 'SGE', 'GTE']
hrv_features = ['HR', 'SDNN', 'RMSSD']
scl_features = ['SCL_mean']

eeg_high = [c for c in X.columns if 'high' in c]
eeg_low = [c for c in X.columns if 'low' in c]
other_features = ['session']


In [18]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('eye', StandardScaler(), eye_features),
        ('hrv', StandardScaler(), hrv_features),
        ('scl', StandardScaler(), scl_features),
        ('eeg_high', Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.9))
        ]), eeg_high),
        ('eeg_low', Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.9))
        ]), eeg_low),
        ('other', StandardScaler(), other_features)
    ]
)


In [19]:
pipe_grouped = Pipeline([
    ('preprocess', preprocessor),
    ('model', Ridge(alpha=1.0))
])

param_grid = {
    'model__alpha': np.logspace(-3, 3, 20)
}

grid_grouped = GridSearchCV(
    pipe_grouped,
    param_grid,
    cv=gkf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_grouped.fit(X, y, groups=groups)

print("最优参数:", grid_grouped.best_params_)
print("CV RMSE:", -grid_grouped.best_score_)

y_pred = cross_val_predict(
    grid_grouped.best_estimator_,
    X, y,
    cv=gkf,
    groups=groups
)

print("被试级 RMSE:", mean_squared_error(y, y_pred, squared=False))
print("R²:", r2_score(y, y_pred))


最优参数: {'model__alpha': 1000.0}
CV RMSE: 1.329990254491567
被试级 RMSE: 1.3370396908756599
R²: -0.01675796386987005


# permutation测试

In [20]:
"""
对每一次 permutation：

对每个被试：
随机打乱该被试 7 个 session 的 label
用 完全相同的 ElasticNet + GroupKFold
计算被试级 R²
重复 1000 次
计算
"""
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

best_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNet(
        alpha=0.12742749857031335,
        l1_ratio=0.9,
        max_iter=10000
    ))
])


In [21]:
gkf = GroupKFold(n_splits=8)


def subject_level_r2(X, y, groups, model):
    y_pred = cross_val_predict(
        model,
        X, y,
        cv=gkf,
        groups=groups
    )
    return r2_score(y, y_pred)


In [22]:
real_r2 = subject_level_r2(X, y, groups, best_model)
print("Real R²:", real_r2)


Real R²: 0.17291208867541064


In [23]:
import numpy as np
from tqdm import trange

n_perm = 1000
perm_r2 = np.zeros(n_perm)

y_array = y.values.copy()
groups_array = groups.values

rng = np.random.default_rng(42)

for i in trange(n_perm):
    y_perm = y_array.copy()

    for subj in np.unique(groups_array):
        idx = np.where(groups_array == subj)[0]
        y_perm[idx] = rng.permutation(y_perm[idx])

    perm_r2[i] = subject_level_r2(X, y_perm, groups, best_model)


 23%|██▎       | 227/1000 [00:05<00:17, 45.07it/s]


KeyboardInterrupt: 

In [ ]:
p_value = (np.sum(perm_r2 >= real_r2) + 1) / (n_perm + 1)
print("Permutation p-value:", p_value)

In [ ]:
# 绘图核心代码（已添加局部字体设置）
plt.hist(perm_r2, bins=30)
plt.axvline(real_r2, color='red', linewidth=2)  # 红色垂直线
# 为每个文本元素单独设置Arial字体
plt.xlabel("Permuted R$^2$", fontfamily='Arial')
plt.ylabel("Count", fontfamily='Arial')
plt.title("Permutation Test", fontfamily='Arial')

# 保存并显示
plt.savefig("Alcohol_permutation_test.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNet(
        alpha=0.12742749857031335,
        l1_ratio=0.9,
        max_iter=10000
    ))
])

In [ ]:
gkf = GroupKFold(n_splits=8)

coef_matrix = []

for train_idx, test_idx in gkf.split(X, y, groups):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    model.fit(X_train, y_train)

    coef = model.named_steps['model'].coef_
    coef_matrix.append(coef)

coef_matrix = np.array(coef_matrix)  # shape: (n_folds, n_features)

In [ ]:
feature_names = X.columns

stability_df = pd.DataFrame({
    'feature': feature_names,
    'selection_freq': np.mean(coef_matrix != 0, axis=0),
    'mean_coef': np.mean(coef_matrix, axis=0),
    'std_coef': np.std(coef_matrix, axis=0),
    'sign_consistency': np.mean(np.sign(coef_matrix) == np.sign(np.mean(coef_matrix)), axis=0)
})

stability_df = stability_df.sort_values('selection_freq', ascending=False)

In [ ]:
print(stability_df)

# 预测图

In [ ]:
from sklearn.model_selection import cross_val_predict

df_subj = cross_val_predict(
    best_model,
    X, y,
    cv=gkf,
    groups=groups
)
n = 16  # 重复次数
session = [i for _ in range(n) for i in range(1, 8)]
# print(session)

df_subj = pd.DataFrame({
    'subject': groups.values,
    'y_true': y.values,
    'y_pred': y_pred,
    'Session': session
})
df_subj

In [ ]:
df_subj_mean = df_subj.drop(columns='Session')
df_subj_mean = df_subj_mean.groupby(["subject"]).mean()
df_subj_mean

In [ ]:
y_true = df_subj_mean['y_true']
y_pred = df_subj_mean['y_pred']
y_pred

In [ ]:
y_true

In [ ]:
plt.scatter(y_true, y_pred, alpha=0.7)
plt.plot([y_true.min(), y_true.max()],
         [y_true.min(), y_true.max()],
         'r--')

plt.xlabel("True Label")
plt.ylabel("Predicted Label")
plt.title("Prediction vs True")
plt.show()